[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maxischa/datacamp_test/blob/main/bloc3_stats/cours/seance3_cours.ipynb)

# Séance 3.3 — Relier deux variables — y a-t-il un lien ?

**Cours** · durée : 2h (≈50 min de cours, ≈50 min d'exercices)

> ⚠️ **Avant de taper quoi que ce soit :** *Fichier → Enregistrer une copie dans Drive*. Sinon votre travail sera perdu en fermant l'onglet.
>
> 📱 Sur tablette, faites d'abord les réglages de [Bien démarrer](https://github.com/maxischa/datacamp_test/blob/main/ressources/setup_tablette.md).

## Objectifs

À la fin de cette séance, vous saurez :

- découper une variable continue en catégories avec `pd.cut`
- tester le lien entre deux variables qualitatives avec un khi-deux
- lire un tableau d'effectifs attendus pour dire *où* est la dépendance
- mesurer un lien entre deux variables quantitatives (Pearson, Spearman)
- reconnaître les trois pièges de la corrélation : extrêmes, non-linéarité, causalité

## Deux questions, deux outils

1. *« Les grosses commandes sont-elles réparties de la même façon selon les
   pays ? »*
2. *« Qu'est-ce qui fait le montant d'une commande ? »*

La première croise deux variables **qualitatives** (le pays, la taille de
commande). La seconde croise deux variables **quantitatives**. Ce ne sont pas
les mêmes outils.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

# Affichage adapte aux petits ecrans
pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 80)

# Les donnees sont lues directement depuis le web : rien a telecharger
BASE = "https://raw.githubusercontent.com/maxischa/datacamp_test/main/bloc3_stats/data/"

In [ ]:
cmd = pd.read_csv(BASE + "commandes.csv")   ## une ligne = une commande

print(cmd.shape)
cmd.head(3)

## 1. Deux variables qualitatives

### D'abord, fabriquer la qualitative

`ca` est un montant. Pour raisonner en catégories, on le découpe.

In [ ]:
# Les bornes sont un choix de gestion, pas une verite mathematique.
# 1e9 = "et tout ce qui est au-dessus".
cmd["taille"] = pd.cut(cmd["ca"], [0, 200, 500, 1e9],   ## 4 bornes, 3 tranches
                       labels=["petite", "moyenne", "grande"])

cmd["taille"].value_counts()   ## combien dans chacune ?

### Ensuite, croiser

On garde les quatre pays les plus présents, pour un tableau lisible.

In [ ]:
top4 = cmd["pays"].value_counts().head(4).index   ## les quatre plus gros
sub = cmd.query("pays in @top4")   ## @ = va chercher la variable Python

tableau = pd.crosstab(sub["pays"], sub["taille"])   ## lignes, puis colonnes
tableau

L'Irlande compte 157 grosses commandes pour 29 petites ; le Royaume-Uni,
183 grosses pour 278 petites. Les profils ont l'air différents.

**« Ont l'air » ne suffit pas.** Ces écarts sont-ils trop grands pour du
hasard ?

In [ ]:
# Quatre resultats : la statistique, la p-value, les degres de
# liberte, et le tableau des effectifs ATTENDUS sous independance
khi2, p, ddl, attendus = stats.chi2_contingency(tableau)

print("p-value :", p)   ## minuscule : la dependance est certaine

**p ≈ 0,000000000000000000000000006.** La répartition des tailles de commande
dépend bel et bien du pays. Ce n'est pas du hasard.

### Mais *où* est la dépendance ?

Le test dit qu'il y en a une. Il ne dit pas laquelle. Pour l'écrire dans une
note, il faut comparer ce qu'on observe à ce qu'on **attendrait** si le pays
n'y était pour rien.

In [ ]:
att = pd.DataFrame(attendus, index=tableau.index, columns=tableau.columns)

(tableau - att).round(1)   ## observe moins attendu : OU est la dependance

Maintenant on peut écrire quelque chose :

- **Irlande : +71,8 grosses commandes** par rapport à l'attendu, et −39,2
  petites. C'est un profil de gros acheteur.
- **Royaume-Uni : −82,7 grosses**, +65,5 petites. Un profil de détail.

> 💡 **Le réflexe.** Un khi-deux significatif n'est jamais une conclusion,
> c'est un feu vert pour aller regarder le tableau des écarts. C'est là
> qu'est l'information.

## 2. Deux variables quantitatives

**On trace d'abord.** Toujours.

In [ ]:
# On trace AVANT de calculer : un coefficient ne montre pas la forme
cmd.plot(kind="scatter", x="qte", y="ca", alpha=0.3, figsize=(7, 4))
plt.title("Montant et nombre d'articles")
plt.xlabel("articles")
plt.ylabel("euros")
plt.show()

Le lien saute aux yeux : plus d'articles, plus d'euros. La **corrélation** met
un nombre sur cette impression.

Elle vaut **1** pour une droite croissante parfaite, **−1** pour une droite
décroissante parfaite, **0** quand il n'y a aucune tendance droite.

In [ ]:
cmd[["ca", "nart", "qte"]].corr().round(3)   ## toutes les paires

- `ca` et `qte` : **0,85**. Très lié.
- `ca` et `nart` : **0,38**. Beaucoup plus faible.

Étonnant : `nart` est le nombre de références différentes dans la commande.
Une commande de 40 références devrait coûter plus cher qu'une commande de 3.

### Pearson et Spearman

La corrélation par défaut (**Pearson**) mesure l'alignement sur une droite.
La corrélation de **Spearman** raisonne sur les **rangs** : elle demande
seulement si l'un monte quand l'autre monte, quelle que soit la forme.

In [ ]:
print("Pearson  :", round(cmd["ca"].corr(cmd["nart"]), 3))   ## une droite
print("Spearman :", round(cmd["ca"].corr(cmd["nart"], method="spearman"), 3))

**0,38 contre 0,67.** Même couple de variables, deux réponses très
différentes. Regardez le nuage pour comprendre :

In [ ]:
cmd.plot(kind="scatter", x="nart", y="ca", alpha=0.3, figsize=(7, 4))
plt.title("Montant et nombre de references")
plt.xlabel("references")
plt.ylabel("euros")
plt.show()

Le lien existe, mais il est **courbe** et une poignée de commandes énormes
étirent l'échelle. Pearson, qui cherche une droite, sous-estime. Spearman, qui
ne regarde que l'ordre, voit le lien réel.

> 💡 **La règle.** Quand Pearson et Spearman divergent nettement, c'est que la
> relation n'est pas droite, ou que des valeurs extrêmes pèsent lourd. Les
> deux cas se voient sur le nuage de points, jamais dans le coefficient.

### Le poids des extrêmes

In [ ]:
seuil = cmd["ca"].quantile(0.99)   ## le montant des 1 % du haut
sans = cmd.query("ca < @seuil")    ## 20 lignes sur 1 955 en moins

print("avec :", round(cmd["ca"].corr(cmd["nart"]), 3))
print("sans :", round(sans["ca"].corr(sans["nart"]), 3), "-", len(cmd) - len(sans), "lignes en moins")

**0,38 → 0,49 en retirant 20 lignes sur 1 955.**

Un coefficient de corrélation cité sans le nuage de points qui va avec n'est
pas un résultat.

## 3. Corrélation n'est pas causalité

### Piège 1 — la tautologie

`qte` et `ca` corrèlent à 0,85. Rien d'étonnant : **le montant d'une commande
se calcule à partir des quantités**. Une variable et un de ses composants
corrèlent toujours, et ça n'apprend rien.

Une corrélation n'est une information que si les deux variables sont mesurées
**indépendamment** l'une de l'autre.

### Piège 2 — le coefficient global qui cache tout

In [ ]:
for pays in ["Belgique", "France", "Royaume-Uni", "Irlande"]:
    d = cmd.query("pays == @pays")   ## un marche a la fois
    print(f"{pays:<13}", round(d["ca"].corr(d["nart"]), 3))

**0,90 en Belgique, 0,23 en Irlande.** Le 0,38 global n'est le chiffre de
personne.

Et l'Irlande, vous savez pourquoi : **deux clients**. Ce n'est pas le pays qui
produit ce comportement d'achat, c'est le type de client. Le pays n'est qu'une
étiquette posée dessus — une **variable de confusion**.

> ⚠️ Avant d'écrire « X influence Y », posez-vous la question : **existe-t-il
> un Z qui expliquerait les deux ?** Ici, Z est « c'est un grossiste ».

## 4. Deux erreurs, dont une qui ne prévient pas

### L'erreur bruyante

In [ ]:
cmd[["pays", "ca"]].corr()   ## "pays" est du texte

Dernière ligne :

```
ValueError: could not convert string to float: 'France'
```

Une corrélation demande deux colonnes de **nombres**. Pour croiser un pays et
un montant, l'outil était celui de la séance 3.2.

### L'erreur silencieuse

Voici deux colonnes où `y` est **entièrement déterminé** par `x` : aucune part
de hasard.

In [ ]:
x = pd.Series(range(-50, 51))
y = x ** 2   ## lien parfait, mais en forme de U

print("correlation :", round(x.corr(y), 3))   ## 0,000 : la droite n'y est pas

**0,000.** Un lien parfait, et une corrélation nulle.

La corrélation ne mesure pas « y a-t-il un lien ». Elle mesure « y a-t-il un
lien **de forme droite** ». Ici, la courbe descend puis remonte : les deux
moitiés s'annulent exactement.

In [ ]:
plt.plot(x, y)
plt.title("Correlation nulle, lien parfait")
plt.show()

Trente secondes de graphique auraient évité la conclusion « ces deux variables
n'ont rien à voir ».

> ⚠️ **Tracez toujours avant de conclure.** C'est la seule règle de cette
> séance qui n'a aucune exception.

---

## Ce que vous savez faire maintenant

| Vous voulez... | La commande |
|---|---|
| découper une variable continue | `pd.cut(df["ca"], [0, 200, 500, 1e9], labels=[...])` |
| croiser deux qualitatives | `pd.crosstab(df["pays"], df["taille"])` |
| tester leur lien | `stats.chi2_contingency(tableau)` |
| les effectifs attendus si indépendance | `stats.chi2_contingency(tableau)[3]` |
| toutes les corrélations d'un coup | `df[["ca", "nart", "qte"]].corr()` |
| une corrélation entre deux colonnes | `df["ca"].corr(df["qte"])` |
| la version robuste (rangs) | `df["ca"].corr(df["qte"], method="spearman")` |
| voir le lien | `df.plot(kind="scatter", x="qte", y="ca")` |

## Choisir son outil

| Variable 1 | Variable 2 | Outil |
|---|---|---|
| qualitative | qualitative | tableau croisé + khi-deux |
| quantitative | quantitative | nuage de points + corrélation |
| qualitative | quantitative | comparaison de moyennes (séance 3.2) |

## Les trois pièges

1. **Les extrêmes.** Retirer 1 % des commandes fait passer la corrélation
   `ca`/`nart` de 0,38 à 0,49. Vingt lignes sur 1 955.
2. **La non-linéarité.** Une corrélation nulle ne veut pas dire « aucun lien ».
   Elle veut dire « aucun lien **de forme droite** ». **Tracez toujours.**
3. **La causalité.** `qte` et `ca` corrèlent à 0,85 — évidemment, `qte` sert à
   calculer `ca`. Une corrélation n'est une information que si les deux
   variables sont mesurées indépendamment l'une de l'autre.